# 01 · Dữ liệu, cohort và nhãn (phụ trách: Hùng · Hà)

**Đề:** *"Build a distributed feature-engineering pipeline for registry or EHR data"* và *"in-hospital mortality, 30-day readmission, or medium-term adverse outcomes"*.

Notebook này đọc **số liệu thật đã kiểm chứng** (`results/data/verified_v1.json`) để bạn nắm chắc từng con số: phễu cohort, ba bộ nhãn, bản chia tập.
Mã nguồn: `data_spark/cohort.py`, `data_spark/labels.py`, cấu hình `configs/hf_codes.yaml`, `configs/data.yaml`.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

## 1. Phễu cohort suy tim
Mỗi bước lọc loại bao nhiêu lượt nhập viện? Vì sao lại có 21 lượt bị loại ở S2?

In [ ]:
pd.DataFrame(VERIFIED['cohort_funnel'])

In [ ]:
VERIFIED['cohort_extra']

## 2. Ba bài toán và ca bị loại
Chú ý: quần thể mỗi bài khác nhau vì **mốc dự đoán** khác nhau (P1: giờ 24; P2/P3: lúc xuất viện).

In [ ]:
lab = pd.DataFrame(VERIFIED['labels']); lab

In [ ]:
pd.DataFrame({r['task']: r['excluded'] for r in VERIFIED['labels']}).T

In [ ]:
# Kiểm tra chéo: số lượt = cohort − bị loại
cohort = 80746
for r in VERIFIED['labels']:
    ex = sum(v for v in r['excluded'].values() if isinstance(v, int))
    print(r['task'], 'cohort − loại =', cohort - ex, '| nhãn =', r['admissions'])
print('P1 còn <10 ca tử vong trước giờ 24 bị che nên lệch ≤ 9')

## 3. 417 ca tử vong nằm ngoài P1
Giải thích được điều này là một điểm cộng khi bảo vệ.

In [ ]:
VERIFIED['p1_deaths_outside_task']

## 4. Tỷ lệ dương theo giai đoạn (nguồn của drift nhãn)

In [ ]:
yg = pd.DataFrame(VERIFIED['by_year_group_p1']); yg['share'] = yg['n'] / yg['n'].sum(); yg

## 5. Bản chia tập đã khóa

In [ ]:
pd.DataFrame({k: v for k, v in VERIFIED['split_p1'].items() if isinstance(v, dict)}).T

## Câu hỏi tự kiểm tra
1. Vì sao chia theo `subject_id` chứ không theo `hadm_id`?
2. Nhãn P2 loại những ca nào, và vì sao loại "chết trong 30 ngày mà không tái nhập"?
3. `dod` của MIMIC có giới hạn gì với P3?
4. Vì sao cohort chọn bằng mã ICD lúc xuất viện lại là vấn đề với P1? Phân tích C2 trong báo cáo trả lời thế nào?

**Chạy thật trên EC2:** `make etl TASK=p1 VERSION=v1_1`, rồi `make etl TASK=p2`, `make etl TASK=p3`.